In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window



##Create a dataframe

In [0]:
data = [
    ("A", "S1", "2024-01-01", 10, 100, 0),
    ("A", "S1", "2024-01-02", 12, 100, 1),
    ("A", "S1", "2024-01-03", 8, 110, 0),
    ("A", "S1", "2024-01-04", 15, 110, 1),
    ("A", "S1", "2024-01-05", 9, 105, 0),
    ("B", "S1", "2024-01-01", 20, 200, 0),
    ("B", "S1", "2024-01-02", 18, 200, 0),
    ("B", "S1", "2024-01-03", 25, 210, 1),
    ("B", "S1", "2024-01-04", 22, 210, 0),
    ("B", "S1", "2024-01-05", 30, 220, 1),
]

columns = ["product_id", "store_id", "date", "units", "price", "promo_flag"]

df = spark.createDataFrame(data, columns)

In [0]:
df.display()

product_id,store_id,date,units,price,promo_flag
A,S1,2024-01-01,10,100,0
A,S1,2024-01-02,12,100,1
A,S1,2024-01-03,8,110,0
A,S1,2024-01-04,15,110,1
A,S1,2024-01-05,9,105,0
B,S1,2024-01-01,20,200,0
B,S1,2024-01-02,18,200,0
B,S1,2024-01-03,25,210,1
B,S1,2024-01-04,22,210,0
B,S1,2024-01-05,30,220,1


In [0]:
df = (
    df.withColumn("date",F.to_date("date"))
    .orderBy('product_id','store_id')
)
df.display()

product_id,store_id,date,units,price,promo_flag
A,S1,2024-01-01,10,100,0
A,S1,2024-01-02,12,100,1
A,S1,2024-01-03,8,110,0
A,S1,2024-01-04,15,110,1
A,S1,2024-01-05,9,105,0
B,S1,2024-01-01,20,200,0
B,S1,2024-01-02,18,200,0
B,S1,2024-01-03,25,210,1
B,S1,2024-01-04,22,210,0
B,S1,2024-01-05,30,220,1


In [0]:
base_window = Window.partitionBy('product_id','store_id').orderBy('date')

### Build autoregressive features (lag 1, lag 2, lag 3)

In [0]:
df = (
  df.withColumn("lag1", F.lag("units",1).over(base_window))
  .withColumn("lag2", F.lag("units",2).over(base_window))
  .withColumn("lag3", F.lag("units",3).over(base_window))
)
df.display()

product_id,store_id,date,units,price,promo_flag,lag1,lag2,lag3
A,S1,2024-01-01,10,100,0,null,null,null
A,S1,2024-01-02,12,100,1,10,null,null
A,S1,2024-01-03,8,110,0,12,10,null
A,S1,2024-01-04,15,110,1,8,12,10
A,S1,2024-01-05,9,105,0,15,8,12
B,S1,2024-01-01,20,200,0,null,null,null
B,S1,2024-01-02,18,200,0,20,null,null
B,S1,2024-01-03,25,210,1,18,20,null
B,S1,2024-01-04,22,210,0,25,18,20
B,S1,2024-01-05,30,220,1,22,25,18


## Build 3-day rolling average demand

In [0]:
rolling_3 = base_window.rowsBetween(-2,0)

In [0]:
df = (
  df.withColumn("rolling_average", F.avg("units").over(rolling_3))
)
df.display()

product_id,store_id,date,units,price,promo_flag,lag1,lag2,lag3,rolling_average
A,S1,2024-01-01,10,100,0,null,null,null,10.0
A,S1,2024-01-02,12,100,1,10,null,null,11.0
A,S1,2024-01-03,8,110,0,12,10,null,10.0
A,S1,2024-01-04,15,110,1,8,12,10,11.666666666666666
A,S1,2024-01-05,9,105,0,15,8,12,10.666666666666666
B,S1,2024-01-01,20,200,0,null,null,null,20.0
B,S1,2024-01-02,18,200,0,20,null,null,19.0
B,S1,2024-01-03,25,210,1,18,20,null,21.0
B,S1,2024-01-04,22,210,0,25,18,20,21.666666666666668
B,S1,2024-01-05,30,220,1,22,25,18,25.666666666666668


## Detect demand spikes (> 1.5x rolling average)

In [0]:
df = (
  df.withColumn(
    "Demand_spike",
    F.when(
      F.col('units') > 1.5 * F.col('rolling_average'),1
  ).otherwise(0)
)
)

In [0]:
df.display()

product_id,store_id,date,units,price,promo_flag,lag1,lag2,lag3,rolling_average,Demand_spike
A,S1,2024-01-01,10,100,0,null,null,null,10.0,0
A,S1,2024-01-02,12,100,1,10,null,null,11.0,0
A,S1,2024-01-03,8,110,0,12,10,null,10.0,0
A,S1,2024-01-04,15,110,1,8,12,10,11.666666666666666,0
A,S1,2024-01-05,9,105,0,15,8,12,10.666666666666666,0
B,S1,2024-01-01,20,200,0,null,null,null,20.0,0
B,S1,2024-01-02,18,200,0,20,null,null,19.0,0
B,S1,2024-01-03,25,210,1,18,20,null,21.0,0
B,S1,2024-01-04,22,210,0,25,18,20,21.666666666666668,0
B,S1,2024-01-05,30,220,1,22,25,18,25.666666666666668,0


## Cumulative promo sales per product

In [0]:
df = (
  df.withColumn(
    "Cumulative_promo_sum",
    F.sum(F.when(F.col("promo_flag")==1, F.col('units')).otherwise(0)).over(base_window)
  )
)

In [0]:
df.display()

product_id,store_id,date,units,price,promo_flag,lag1,lag2,lag3,rolling_average,Demand_spike,Cumulative_promo_sum
A,S1,2024-01-01,10,100,0,null,null,null,10.0,0,0
A,S1,2024-01-02,12,100,1,10,null,null,11.0,0,12
A,S1,2024-01-03,8,110,0,12,10,null,10.0,0,12
A,S1,2024-01-04,15,110,1,8,12,10,11.666666666666666,0,27
A,S1,2024-01-05,9,105,0,15,8,12,10.666666666666666,0,27
B,S1,2024-01-01,20,200,0,null,null,null,20.0,0,0
B,S1,2024-01-02,18,200,0,20,null,null,19.0,0,0
B,S1,2024-01-03,25,210,1,18,20,null,21.0,0,25
B,S1,2024-01-04,22,210,0,25,18,20,21.666666666666668,0,25
B,S1,2024-01-05,30,220,1,22,25,18,25.666666666666668,0,55


## Rank top sales day per product

In [0]:
rank_window = Window.partitionBy('product_id').orderBy(F.desc('units'))

In [0]:
df = (
    df.withColumn("sales_rank",
    F.rank().over(rank_window)
)
)
df.filter("sales_rank==1").select('product_id','date','units').display()

product_id,date,units
A,2024-01-04,15
B,2024-01-05,30


## Percent change from previous day

In [0]:
df = (
    df.withColumn(
        "%change",
        (F.col("units") - F.col("lag1"))/F.col("lag1")
    )
)
df.display()

product_id,store_id,date,units,price,promo_flag,lag1,lag2,lag3,rolling_average,Demand_spike,Cumulative_promo_sum,sales_rank,%change
A,S1,2024-01-04,15,110,1,8,12,10,11.666666666666666,0,27,1,0.875
A,S1,2024-01-02,12,100,1,10,null,null,11.0,0,12,2,0.2
A,S1,2024-01-01,10,100,0,null,null,null,10.0,0,0,3,null
A,S1,2024-01-05,9,105,0,15,8,12,10.666666666666666,0,27,4,-0.4
A,S1,2024-01-03,8,110,0,12,10,null,10.0,0,12,5,-0.3333333333333333
B,S1,2024-01-05,30,220,1,22,25,18,25.666666666666668,0,55,1,0.36363636363636365
B,S1,2024-01-03,25,210,1,18,20,null,21.0,0,25,2,0.3888888888888889
B,S1,2024-01-04,22,210,0,25,18,20,21.666666666666668,0,25,3,-0.12
B,S1,2024-01-01,20,200,0,null,null,null,20.0,0,0,4,null
B,S1,2024-01-02,18,200,0,20,null,null,19.0,0,0,5,-0.1


## Rolling Standard Deviation (Volatility Feature)

In [0]:
df = (
  df.withColumn(
    "Rolling_std_Deviation",
    F.stddev("units").over(rolling_3)
  )
)
df.display()

product_id,store_id,date,units,price,promo_flag,lag1,lag2,lag3,rolling_average,Demand_spike,Cumulative_promo_sum,sales_rank,%change,Rolling_std_Deviation
A,S1,2024-01-01,10,100,0,null,null,null,10.0,0,0,3,null,null
A,S1,2024-01-02,12,100,1,10,null,null,11.0,0,12,2,0.2,1.4142135623730951
A,S1,2024-01-03,8,110,0,12,10,null,10.0,0,12,5,-0.3333333333333333,2.0
A,S1,2024-01-04,15,110,1,8,12,10,11.666666666666666,0,27,1,0.875,3.511884584284246
A,S1,2024-01-05,9,105,0,15,8,12,10.666666666666666,0,27,4,-0.4,3.7859388972001824
B,S1,2024-01-01,20,200,0,null,null,null,20.0,0,0,4,null,null
B,S1,2024-01-02,18,200,0,20,null,null,19.0,0,0,5,-0.1,1.4142135623730951
B,S1,2024-01-03,25,210,1,18,20,null,21.0,0,25,2,0.3888888888888889,3.605551275463989
B,S1,2024-01-04,22,210,0,25,18,20,21.666666666666668,0,25,3,-0.12,3.5118845842842465
B,S1,2024-01-05,30,220,1,22,25,18,25.666666666666668,0,55,1,0.36363636363636365,4.041451884327381


## Days Since Last Promo

In [0]:
cumulative_window = base_window.rowsBetween(
    Window.unboundedPreceding, Window.currentRow
)

In [0]:
df = (
    df.withColumn("promo_day",
    F.when(F.col("promo_flag")==1, F.col("date"))
)
)

df = (
    df.withColumn("last_promo_date",
    F.last("promo_day",ignorenulls=True).over(cumulative_window)
)
)
df = (
    df.withColumn("days_since_last_promo",
    F.datediff(F.col("date"), F.col("last_promo_date"))
)
)
df.display()


product_id,store_id,date,units,price,promo_flag,lag1,lag2,lag3,rolling_average,Demand_spike,Cumulative_promo_sum,sales_rank,%change,Rolling_std_Deviation,promo_day,last_promo_date,days_since_last_promo
A,S1,2024-01-01,10,100,0,null,null,null,10.0,0,0,3,null,null,null,null,null
A,S1,2024-01-02,12,100,1,10,null,null,11.0,0,12,2,0.2,1.4142135623730951,2024-01-02,2024-01-02,0
A,S1,2024-01-03,8,110,0,12,10,null,10.0,0,12,5,-0.3333333333333333,2.0,null,2024-01-02,1
A,S1,2024-01-04,15,110,1,8,12,10,11.666666666666666,0,27,1,0.875,3.511884584284246,2024-01-04,2024-01-04,0
A,S1,2024-01-05,9,105,0,15,8,12,10.666666666666666,0,27,4,-0.4,3.7859388972001824,null,2024-01-04,1
B,S1,2024-01-01,20,200,0,null,null,null,20.0,0,0,4,null,null,null,null,null
B,S1,2024-01-02,18,200,0,20,null,null,19.0,0,0,5,-0.1,1.4142135623730951,null,null,null
B,S1,2024-01-03,25,210,1,18,20,null,21.0,0,25,2,0.3888888888888889,3.605551275463989,2024-01-03,2024-01-03,0
B,S1,2024-01-04,22,210,0,25,18,20,21.666666666666668,0,25,3,-0.12,3.5118845842842465,null,2024-01-03,1
B,S1,2024-01-05,30,220,1,22,25,18,25.666666666666668,0,55,1,0.36363636363636365,4.041451884327381,2024-01-05,2024-01-05,0


## Top 2 sales days per product

In [0]:
df.filter(F.col("sales_rank").isin(1,2)).select('product_id','date','units').display()

product_id,date,units
A,2024-01-04,15
A,2024-01-02,12
B,2024-01-05,30
B,2024-01-03,25


## Lets exetend the dataset 

In [0]:
data = [
    ("A", "S1", "2024-01-01", 10, 100, 0),
    ("A", "S1", "2024-01-02", 12, 100, 1),
    ("A", "S1", "2024-01-03", 8, 110, 0),
    ("A", "S1", "2024-01-04", 15, 110, 1),
    ("A", "S1", "2024-01-05", 9, 105, 0),
    ("B", "S1", "2024-01-01", 20, 200, 0),
    ("B", "S1", "2024-01-02", 18, 200, 0),
    ("B", "S1", "2024-01-03", 25, 210, 1),
    ("B", "S1", "2024-01-04", 22, 210, 0),
    ("B", "S1", "2024-01-05", 30, 220, 1),
]

columns = ["product_id", "store_id", "date", "units", "price", "promo_flag"]

df1 = spark.createDataFrame(data, columns)

extended_data = [
    ("A", "S2", "2024-01-01", 5, 100, 0),
    ("A", "S2", "2024-01-03", 7, 110, 1),
    ("B", "S2", "2024-01-01", 15, 200, 0),
    ("B", "S2", "2024-01-04", 18, 210, 1),
]

df2 = spark.createDataFrame(extended_data, columns)
df2 = df2.withColumn("date", F.to_date("date"))

df1 = df1.union(df2).orderBy("product_id", "store_id", "date")

In [0]:
df1.display()

product_id,store_id,date,units,price,promo_flag
A,S1,2024-01-01,10,100,0
A,S1,2024-01-02,12,100,1
A,S1,2024-01-03,8,110,0
A,S1,2024-01-04,15,110,1
A,S1,2024-01-05,9,105,0
A,S2,2024-01-01,5,100,0
A,S2,2024-01-03,7,110,1
B,S1,2024-01-01,20,200,0
B,S1,2024-01-02,18,200,0
B,S1,2024-01-03,25,210,1


## Cross-Store Ranking - Which store performs best for each product?

In [0]:
df_agg = df1.groupBy('store_id','product_id').agg(F.sum('units'))

In [0]:
store_rank = Window.partitionBy("store_id").orderBy(F.col("sum(units)"))
df_agg = (
    df_agg.withColumn('store_rank',
                      F.rank().over(store_rank))
)
df_agg.display()

store_id,product_id,sum(units),store_rank
S1,A,54,1
S1,B,115,2
S2,A,12,1
S2,B,33,2
